# Trial-outcome PSTH for QC-passed units on selected probes

Trial-outcome PSTH comparisons for **all units that pass the default QC** and
are recorded on the probes you choose. Units are selected from the NWB using the
`units["device_name"]` (probe) column and `get_units_passed_default_qc`.

Pick the comparison once via the `CONTRAST` switch in section 5:

- `"response"` → **response** (`animal_response != 2`) vs **no-response** (`== 2`)
- `"reward"` → **reward** (either side rewarded) vs **no-reward** (responded, no reward)

The same downstream analyses then run for whichever contrast you chose:

1. Per-unit PSTH (group A vs group B)
2. Population-average PSTH
3. Per-unit scatter of windowed mean rate
4. Paired Wilcoxon signed-rank test + difference histogram
5. Scatter colored by signed difference
6. Trial-count-matched control (subsample the larger group)

To restrict to specific probes, set `PROBES` in the config cell (section 2) to a
list such as `["ProbeA"]` or `["ProbeA", "ProbeB"]` (use the probe names printed
by section 4); leave it `None` for all probes. Figures go to a per-probe,
per-contrast subfolder so different selections never overwrite earlier results.

In [1]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


In [ ]:
# =============================================================================
# 2. CONFIG
# =============================================================================
import re
from pathlib import Path

# --- Session identifiers -----------------------------------------------------
session_date = '863973_2026-08-25_14-18-29'
ephys_session = None   # None -> auto-discover from the PSTH Zarr in step 3

# --- Paths -------------------------------------------------------------------
PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")

# --- Probe selection ---------------------------------------------------------
# Analyze all units that pass the default QC and were recorded on these probes.
# Probes use the NWB units "device_name" values (e.g. "ProbeA", "ProbeB"); set
# PROBES to a list such as ["ProbeA"] or ["ProbeA", "ProbeB"]. Use None (or an
# empty list) for all probes. Section 4 prints the probe names available here.
PROBES = None   # e.g. ["ProbeA"] or ["ProbeA", "ProbeB"]

# Output folder is tagged by the probe selection so runs don't overwrite each other.
_probe_tag = "all_probes" if not PROBES else "probe_" + "_".join(
    re.sub(r"[^0-9A-Za-z._-]", "", str(p)) for p in PROBES
)
OUTDIR = Path(f"/root/capsule/scratch/raster_plot_qc_probes/{session_date}/{_probe_tag}")

# --- Raster / PSTH alignment -------------------------------------------------
ALIGN_TO = "go_cue"
TIME_WIN = (-3, 4)

print(f"Probes     : {'all' if not PROBES else PROBES}")
print(f"Output dir : {OUTDIR}")


Metrics CSV : /root/capsule/scratch/opto_tagging/metrics_863973_2026-08-25_14-18-29.csv
Output dir  : /root/capsule/scratch/raster_plot_tagged/863973_2026-08-25_14-18-29


In [ ]:
# =============================================================================
# 3. LOAD SESSION RESOURCES (Zarr PSTH, behavior summary, NWB, trials)
# =============================================================================
import numpy as np
from general_utils import smart_read_csv
from behavior_utils import find_trials, get_fitted_model_names
from nwb_utils import NWBUtils
from create_psth import load_zarr

# Auto-discover the full sorted session name from the PSTH Zarr if not set.
if ephys_session is None:
    cands = sorted(PSTH_DIR.glob(f"ecephys_{session_date}_sorted*_0.2s.zarr"))
    if not cands:
        raise FileNotFoundError(
            f"No PSTH Zarr matching ecephys_{session_date}_sorted*_0.2s.zarr in {PSTH_DIR}"
        )
    ephys_session = cands[0].name[: -len("_0.2s.zarr")]
print(f"ephys_session: {ephys_session}")

zarr_path = PSTH_DIR / f"{ephys_session}_0.2s.zarr"
beh_csv = RESULTS_DIR / f"behavior_summary-{ephys_session}.csv"
assert zarr_path.exists(), f"Zarr not found: {zarr_path}"
assert beh_csv.exists(), f"Behavior CSV not found: {beh_csv}"

nwb, _ = NWBUtils.combine_nwb(session_name=ephys_session)
psth = load_zarr(str(zarr_path))
beh = smart_read_csv(str(beh_csv))
trials = np.asarray(find_trials(nwb, "response"))
print(f"n_trials (response): {len(trials)}")

# Optogenetics (laser-on) trials -> excluded from the raster/PSTH by default.
if "laser_on_trial" in nwb.trials.colnames:
    laser_on = np.asarray(nwb.trials["laser_on_trial"][:])
    opto_trial_ids = np.where(laser_on == 1)[0].astype(np.int64)
else:
    opto_trial_ids = np.array([], dtype=np.int64)
print(f"opto (laser-on) trials: {len(opto_trial_ids)}")

import matplotlib.pyplot as plt
OUTDIR.mkdir(parents=True, exist_ok=True)


Condition (laser_name ignored): (5.0, 1.0, 'Laser_1', 1.0, 10.0, 0.05) | pulse_index=-1
Tagged units (122): [310, 311, 748, 752, 759, 788, 792, 2023, 2025, 2036, 2059, 2109, 2115, 2116, 2149, 2185, 2200, 2203, 2208, 2209, 2212, 2215, 2225, 2230, 2233, 2238, 2239, 2240, 2241, 2245, 2250, 2255, 2257, 2259, 2263, 2264, 2272, 2277, 2284, 2288, 2289, 2296, 2300, 2303, 2314, 2315, 2317, 2322, 2324, 2326, 2331, 2334, 2344, 2366, 2558, 2586, 2587, 2588, 2609, 2613, 2629, 2630, 2634, 2685, 2689, 2692, 2706, 2732, 2735, 2803, 2811, 2826, 2845, 2946, 2947, 2960, 2983, 3000, 3014, 3020, 3023, 3031, 3032, 3033, 3258, 3269, 3326, 3329, 3335, 3336, 3385, 3393, 3400, 3403, 3453, 3467, 3502, 3603, 3620, 3629, 3656, 3663, 3665, 3673, 3696, 3700, 3705, 3718, 3736, 3757, 3809, 3814, 4045, 4062, 4077, 4106, 4159, 4227, 4270, 4273, 4280, 4320]


In [ ]:
# =============================================================================
# 4. SELECT UNITS: all QC-passed units on the chosen probes (from the NWB)
# =============================================================================
from ephys_behavior import get_units_passed_default_qc

# Probe (device) name per unit, straight from the NWB units table. The row index
# here matches the PSTH "unit_index" coordinate.
device_names = np.asarray(nwb.units["device_name"][:]).astype(str)
avail_probes = sorted(set(device_names))
print(f"Probes in this session: {avail_probes}")

# Units passing the automated default QC (presence/isi/amplitude + not 'noise').
qc_units = np.asarray(get_units_passed_default_qc(nwb), dtype=np.int64)
print(f"QC-passed units: {qc_units.size}")

# Restrict to the requested probes (PROBES=None -> keep all probes).
if PROBES:
    want = [str(p) for p in PROBES]
    missing = [p for p in want if p not in avail_probes]
    if missing:
        print(f"WARNING: requested probes not present: {missing}")
    on_probe = np.where(np.isin(device_names, want))[0].astype(np.int64)
else:
    on_probe = np.arange(device_names.size, dtype=np.int64)

unit_ids = sorted(int(u) for u in np.intersect1d(qc_units, on_probe))

# Keep only units that actually exist in the PSTH store.
try:
    psth_units = {int(u) for u in np.asarray(psth.coords["unit_index"].values)}
except Exception:
    psth_units = None
if psth_units is not None:
    dropped = [u for u in unit_ids if u not in psth_units]
    unit_ids = [u for u in unit_ids if u in psth_units]
    if dropped:
        head = dropped[:10]
        print(f"{len(dropped)} QC units absent from PSTH store, dropped: "
              f"{head}{'...' if len(dropped) > 10 else ''}")

# Per-probe counts for the final selection.
sel_probes = device_names[unit_ids] if len(unit_ids) else np.array([], dtype=str)
counts = {p: int(np.sum(sel_probes == p)) for p in sorted(set(sel_probes))}
print(f"Selected probes: {'all' if not PROBES else PROBES}")
print(f"Selected units ({len(unit_ids)}): {unit_ids}")
print(f"Units per probe: {counts}")
if not unit_ids:
    raise ValueError(
        "No QC-passed units found for this probe selection. Check PROBES against "
        "the probe names printed above (device_name values)."
    )


ephys_session: ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54
Found ephys NWB: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54/nwb/ecephys_863973_2026-08-25_14-18-29_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54/nwb/ecephys_863973_2026-08-25_14-18-29_experiment1_recording1.nwb
Found behavior NWB: /root/capsule/data/behavior_nwb/863973_2026-08-25_14-18-29.nwb
Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/863973_2026-08-25_14-18-29.nwb
Successfully appended units table to behavior NWB.
n_trials (response): 420
opto (laser-on) trials: 86


# 5. Choose the contrast

Pick **which trial-outcome comparison** every analysis below uses by setting
`CONTRAST` in the next cell:

- `"response"` → **response** (`animal_response != 2`) vs **no-response** (`== 2`)
- `"reward"` → **reward** (either side rewarded) vs **no-reward** (responded, no
  reward; no-response trials excluded)

Re-run from this cell downward after changing `CONTRAST`. Figures are written to
a per-contrast subfolder (`response_vs_noresponse/` or `reward_vs_noreward/`) and
filenames carry the contrast tag, so switching does not overwrite prior results.

In [ ]:
# =============================================================================
# 5. CONTRAST SELECTION  (choose which trial-outcome comparison to analyze)
# =============================================================================
# Set CONTRAST to one of:
#   "response" -> response (animal_response != 2)  vs  no-response (== 2)
#   "reward"   -> rewarded (either side)           vs  unrewarded (responded, no reward)
# Every analysis below is driven by this one choice (see trial_outcome_psth.py).
from trial_outcome_psth import resolve_contrast

CONTRAST = "response"   # "response" or "reward"
contrast = resolve_contrast(CONTRAST)

CONTRAST_OUTDIR = OUTDIR / contrast.tag
CONTRAST_OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"CONTRAST = {contrast.name!r}:  {contrast.label_a} ({contrast.type_a})  vs  "
      f"{contrast.label_b} ({contrast.type_b})")
print(f"Output subdir: {CONTRAST_OUTDIR}")


# 6. Per-unit PSTH: group A vs group B (selected units)

For each selected unit, overlay the mean ± SEM PSTH for the two groups of the
selected `CONTRAST` (group A vs group B), aligned to the same event/window as the
latent rasters. Optogenetics (laser-on) trials are excluded by default.

In [ ]:
# =============================================================================
# 6. PER-UNIT PSTH: group A vs group B (selected units only)
# =============================================================================
from trial_outcome_psth import plot_perunit_psth

plot_perunit_psth(
    psth, nwb, unit_ids, contrast,
    align_to_event=ALIGN_TO, time_window=TIME_WIN,
    opto_trial_ids=opto_trial_ids, outdir=CONTRAST_OUTDIR, show=False,
)


# 7. Population-average PSTH: group A vs group B (all selected units)

A single figure overlaying the **mean ± SEM PSTH across all selected units** for
the two groups of the selected `CONTRAST`. For each unit we first average across
trials within a group, then average those per-unit means across units (SEM is
computed across units). Optogenetics (laser-on) trials are excluded by default.

In [ ]:
# =============================================================================
# 7. POPULATION-AVERAGE PSTH: group A vs group B (all selected units)
# =============================================================================
from trial_outcome_psth import load_population_psth, build_groups, plot_population_psth

EXCLUDE_OPTO = True   # drop optogenetics (laser-on) trials from both groups

# Load the PSTH once and build the two trial groups; reused by sections 8-11.
pop = load_population_psth(psth, unit_ids, ALIGN_TO, TIME_WIN)
groups = build_groups(nwb, contrast, pop.avail_ids, opto_trial_ids, exclude_opto=EXCLUDE_OPTO)

plot_population_psth(
    pop, groups, contrast, align_to_event=ALIGN_TO,
    exclude_opto=EXCLUDE_OPTO, opto_n=len(opto_trial_ids),
    save_path=OUTDIR / f"population_psth_{contrast.tag}.png",
)


# 8. Scatter: group A vs group B mean rate (all selected units)

Per-unit scatter of the **mean firing rate within a selected time window** for
the two groups of the selected `CONTRAST`. Each point is one selected unit
(x = group A, y = group B). The dashed line is unity (y = x); points below it
fire more on group-A trials. Optogenetics (laser-on) trials are excluded by
default.

In [ ]:
# =============================================================================
# 8. SCATTER: group A vs group B windowed mean rate (all selected units)
# =============================================================================
from trial_outcome_psth import window_rates, plot_scatter

# Time window (seconds, relative to ALIGN_TO) over which to average the rate.
SCATTER_WIN = (0.0, 0.5)

# Per-unit mean rate in the window for each group (x = group A, y = group B).
x, y = window_rates(pop, groups, contrast, SCATTER_WIN)

plot_scatter(
    x, y, contrast, SCATTER_WIN, n_units=pop.n_units,
    exclude_opto=EXCLUDE_OPTO, opto_n=len(opto_trial_ids),
    save_path=OUTDIR / f"scatter_{contrast.tag}.png",
)


## 9. Paired per-unit difference (group B − group A)

Per unit, take the mean rate over `SCATTER_WIN` on each group, compute the signed
difference `B − A`, test it across units with a Wilcoxon signed-rank test, and
show the distribution of differences. A positive median indicates units fire
*more* on group-B trials in this window. Optogenetics (laser-on) trials follow
the `EXCLUDE_OPTO` toggle from the cells above.

In [ ]:
# =============================================================================
# 9. Paired per-unit difference (group B - group A) + Wilcoxon signed-rank
# =============================================================================
from trial_outcome_psth import paired_diff, print_paired_diff, plot_paired_diff

stats = paired_diff(x, y)      # signed diff B - A per unit + Wilcoxon test
print_paired_diff(stats, contrast)

plot_paired_diff(
    stats, contrast, SCATTER_WIN,
    exclude_opto=EXCLUDE_OPTO, opto_n=len(opto_trial_ids),
    save_path=OUTDIR / f"paired_diff_{contrast.tag}.png",
)


## 10. Scatter colored by signed difference

Same group-A-vs-group-B scatter as section 8, but each unit is colored by its
signed difference `B − A` (diverging colormap centered at 0). Points above the
unity line are reddish (higher on group B), points below are bluish (higher on
group A), making any systematic shift easy to see.

In [ ]:
# =============================================================================
# 10. Scatter colored by signed difference (group B - group A)
# =============================================================================
from trial_outcome_psth import plot_scatter_colored

plot_scatter_colored(
    x, y, stats["diff"], contrast, SCATTER_WIN, n_valid=stats["n_valid"],
    exclude_opto=EXCLUDE_OPTO, opto_n=len(opto_trial_ids),
    save_path=OUTDIR / f"scatter_{contrast.tag}_colored_by_diff.png",
)


## 11. Trial-count matched comparison

The two groups usually have very different trial counts, so the smaller group's
mean is estimated from far fewer trials. Here we repeatedly subsample the
**larger** group down to the **smaller** group's count, recompute the population
PSTH each time, and average across repeats. The matched larger-group PSTH is
plotted against the (fixed) smaller-group PSTH so any remaining separation is not
an artifact of unequal trial counts.

In [ ]:
# =============================================================================
# 11. Trial-count matched comparison (subsample the larger group to the smaller)
# =============================================================================
from trial_outcome_psth import trialcount_matched

res, _ = trialcount_matched(
    pop, groups, contrast, SCATTER_WIN, align_to_event=ALIGN_TO,
    n_boot=200, seed=0, exclude_opto=EXCLUDE_OPTO, opto_n=len(opto_trial_ids),
    save_path=OUTDIR / f"population_psth_trialcount_matched_{contrast.tag}.png",
)

print(f"matched n per group            : {res['n_match']}")
print(f"{res['small_label']} window rate       : {res['small_win']:.3f} Hz")
print(f"{res['big_label']} window rate (match) : {res['big_win_mean']:.3f} +/- {res['big_win_std']:.3f} Hz")
print(f"P({res['big_label']} > {res['small_label']})   : {res['frac_big_higher']:.3f}  ({res['n_boot']} subsamples)")
